In [1]:
"""
fishery_simulation_fixed.py
==========================
Numerical simulation and case study for a density-dependent size-structured
fishery model with exogenous recruitment.

This version fixes the main issues in the original script:
  1. Ensures g(E,l) > 0 on the modeled interval by choosing L_inf > l_m.
  2. Uses a truncated objective J_T over [0, T] instead of reusing the
     infinite-horizon functional numerically.
  3. Uses consistent unit comments for alpha, mu1, p_flux, and m0.
  4. Avoids unsupported claims such as monotone decrease of the stationary profile.
  5. Produces explicit numerical outputs (CSV + console summary) including
     E*, N*, R(E*), and l*_opt.
  6. Uses Brent's bracketing method correctly and computes dt from a CFL rule.

Outputs:
  - fig_steady_state_profile.pdf
  - fig_replacement_index.pdf
  - fig_time_dynamics.pdf
  - fig_revenue_threshold.pdf
  - fig_profile_comparison.pdf
  - numerical_summary.csv
  - threshold_sweep.csv

Run:
  python fishery_simulation_fixed.py
"""

from __future__ import annotations

import os
from dataclasses import dataclass, asdict
from typing import Callable, Dict, List, Optional, Tuple

import numpy as np
from scipy.integrate import simpson
from scipy.optimize import brentq
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from cycler import cycler

# ============================================================
# 0. GLOBAL PLOTTING SETTINGS (Publication Quality)
# ============================================================
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 12,
    "axes.labelsize": 13,
    "axes.titlesize": 14,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "lines.linewidth": 2.0,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
    # Academic colorblind-friendly palette
    "axes.prop_cycle": cycler('color', ['#0072B2', '#D55E00', '#009E73', '#CC79A7', '#F0E442', '#56B4E9']),
    "figure.dpi": 300,
    "savefig.bbox": "tight"
})

# ============================================================
# 1. PARAMETERS
# ============================================================

@dataclass
class Params:
    # Size domain (cm)
    l0: float = 20.0
    lm: float = 130.0

    # Growth parameters (cod-like; choose L_inf > l_m so g > 0 on [l0, lm])
    L_inf: float = 135.3              # cm
    K_vb: float = 0.17               # yr^-1
    alpha: float = 5.0e-6            # individual^-1

    # Natural mortality
    mu0: float = 0.20                # yr^-1
    mu1: float = 1.0e-7              # yr^-1 individual^-1

    # Crowding kernel chi(l) = chi_coeff * l^2
    chi_coeff: float = 1.0e-4        # cm^-2

    # Market value c(l) = c0 * l^3
    c0: float = 1.0e-5               # $ cm^-3

    # Replacement index fertility m(l)
    m0: float = 2.0                  # yr^-1
    l_mat: float = 50.0              # cm

    # Exogenous recruitment flux
    p_flux: float = 5.0e4            # individuals yr^-1

    # Economic / regulatory
    r_disc: float = 0.05             # yr^-1
    u_max: float = 0.50              # yr^-1
    T_horizon: float = 60.0          # yr

    # Numerical discretization
    Nl: int = 400                    # number of spatial cells
    cfl_safety: float = 0.8          # CFL safety factor
    record_interval: float = 0.25    # yr
    threshold_count: int = 32        # sweep resolution

    # Output
    output_dir: str = "/mnt/data/fishery_outputs"


P = Params()


def build_grid(p: Params) -> Tuple[np.ndarray, np.ndarray, float]:
    dl = (p.lm - p.l0) / p.Nl
    l_edges = np.linspace(p.l0, p.lm, p.Nl + 1)
    l_cells = 0.5 * (l_edges[:-1] + l_edges[1:])
    return l_edges, l_cells, dl


L_EDGES, L_CELLS, DL = build_grid(P)


def max_growth_bound(p: Params) -> float:
    """Upper bound on g(E,l) over the domain and admissible E>=0."""
    return p.K_vb * (p.L_inf - p.l0)


DT = P.cfl_safety * DL / max_growth_bound(P)
NT = int(np.ceil(P.T_horizon / DT))


# ============================================================
# 2. MODEL FUNCTIONS
# ============================================================

def g_fun(E: float | np.ndarray, l: np.ndarray | float, p: Params = P) -> np.ndarray:
    return p.K_vb * (p.L_inf - np.asarray(l, dtype=float)) / (1.0 + p.alpha * E)


def mu_fun(E: float | np.ndarray, l: np.ndarray | float, p: Params = P) -> np.ndarray:
    arr = np.asarray(l, dtype=float)
    return (p.mu0 + p.mu1 * E) * np.ones_like(arr)


def chi_fun(l: np.ndarray | float, p: Params = P) -> np.ndarray:
    return p.chi_coeff * np.asarray(l, dtype=float) ** 2


def c_fun(l: np.ndarray | float, p: Params = P) -> np.ndarray:
    return p.c0 * np.asarray(l, dtype=float) ** 3


def m_fun(l: np.ndarray | float, p: Params = P) -> np.ndarray:
    arr = np.asarray(l, dtype=float)
    return np.where(arr >= p.l_mat, p.m0 * (arr / p.lm) ** 3, 0.0)


def crowding_index(x: np.ndarray, l: np.ndarray = L_CELLS, p: Params = P) -> float:
    return float(simpson(chi_fun(l, p) * x, x=l))


def total_population(x: np.ndarray, l: np.ndarray = L_CELLS) -> float:
    return float(simpson(x, x=l))


def cumulative_trapezoid_nonuniform(y: np.ndarray, x: np.ndarray) -> np.ndarray:
    out = np.zeros_like(y, dtype=float)
    if len(y) <= 1:
        return out
    dx = np.diff(x)
    out[1:] = np.cumsum(0.5 * (y[:-1] + y[1:]) * dx)
    return out


# ============================================================
# 3. STATIONARY PROFILES AND CLOSURE
# ============================================================

def stationary_profile(E: float, l_arr: np.ndarray, u_arr: Optional[np.ndarray] = None,
                       p: Params = P) -> np.ndarray:
    if u_arr is None:
        u_arr = np.zeros_like(l_arr)
    g_vals = g_fun(E, l_arr, p)
    if np.any(g_vals <= 0.0):
        raise ValueError("Growth rate became nonpositive on the modeled interval.")
    hazard = (mu_fun(E, l_arr, p) + u_arr) / g_vals
    cum_hazard = cumulative_trapezoid_nonuniform(hazard, l_arr)
    return (p.p_flux / g_vals) * np.exp(-cum_hazard)


def closure_F(E: float, l_arr: np.ndarray, u_arr: Optional[np.ndarray] = None,
              p: Params = P) -> float:
    x = stationary_profile(E, l_arr, u_arr, p)
    rhs = simpson(chi_fun(l_arr, p) * x, x=l_arr)
    return float(E - rhs)


def find_bracket(func: Callable[[float], float], start: Tuple[float, float] = (1e-6, 1.0),
                 growth: float = 10.0, max_expand: int = 20) -> Tuple[float, float]:
    a, b = start
    fa, fb = func(a), func(b)
    for _ in range(max_expand):
        if np.sign(fa) != np.sign(fb):
            return a, b
        b *= growth
        fb = func(b)
    raise ValueError("Could not bracket a root for the closure equation.")


def find_Estar(l_arr: np.ndarray, u_arr: Optional[np.ndarray] = None, p: Params = P) -> float:
    f = lambda E: closure_F(E, l_arr, u_arr, p)
    bracket = find_bracket(f)
    return float(brentq(f, bracket[0], bracket[1], xtol=1e-10, rtol=1e-10, maxiter=500))


# ============================================================
# 4. REPLACEMENT INDEX
# ============================================================

def replacement_index(E: float, l_arr: np.ndarray, p: Params = P) -> float:
    g_vals = g_fun(E, l_arr, p)
    hazard = mu_fun(E, l_arr, p) / g_vals
    cum_hazard = cumulative_trapezoid_nonuniform(hazard, l_arr)
    integrand = (m_fun(l_arr, p) / g_vals) * np.exp(-cum_hazard)
    return float(simpson(integrand, x=l_arr))


# ============================================================
# 5. PDE SOLVER AND TRUNCATED OBJECTIVE J_T
# ============================================================

def threshold_control(l_threshold: float, l: np.ndarray, p: Params = P) -> np.ndarray:
    return np.where(l > l_threshold, p.u_max, 0.0)


def solve_pde(l_threshold: float,
              T_end: float,
              x_init: Optional[np.ndarray] = None,
              p: Params = P,
              record: bool = True,
              record_interval: Optional[float] = None) -> Dict[str, object]:
    if record_interval is None:
        record_interval = p.record_interval

    u_profile = threshold_control(l_threshold, L_CELLS, p)

    if x_init is None:
        E0 = find_Estar(L_CELLS, None, p)
        x = stationary_profile(E0, L_CELLS, None, p).copy()
    else:
        x = np.asarray(x_init, dtype=float).copy()

    Nt = int(np.ceil(T_end / DT))
    times: List[float] = []
    x_series: List[np.ndarray] = []
    E_series: List[float] = []
    N_series: List[float] = []
    J_T = 0.0
    next_record = 0.0

    for n in range(Nt + 1):
        t = min(n * DT, T_end)
        E = crowding_index(x, L_CELLS, p)
        N = total_population(x, L_CELLS)

        if record and t >= next_record - 1e-12:
            times.append(t)
            x_series.append(x.copy())
            E_series.append(E)
            N_series.append(N)
            next_record += record_interval

        instant_revenue = float(simpson(c_fun(L_CELLS, p) * u_profile * x, x=L_CELLS))
        dt_step = DT if t < T_end else 0.0
        if n < Nt:
            J_T += np.exp(-p.r_disc * t) * instant_revenue * dt_step

        if n == Nt:
            break

        g_cent = g_fun(E, L_CELLS, p)
        mu_cent = mu_fun(E, L_CELLS, p)

        flux = np.zeros(p.Nl + 1)
        flux[0] = p.p_flux
        g_edges = g_fun(E, L_EDGES, p)
        flux[1:] = g_edges[1:] * x

        x_new = x - (DT / DL) * (flux[1:] - flux[:-1]) - DT * (mu_cent + u_profile) * x
        np.maximum(x_new, 0.0, out=x_new)
        x = x_new

    return {
        "times": np.array(times),
        "x_series": x_series,
        "E_series": np.array(E_series),
        "N_series": np.array(N_series),
        "x_final": x,
        "E_final": crowding_index(x, L_CELLS, p),
        "N_final": total_population(x, L_CELLS),
        "J_T": float(J_T),
    }


# ============================================================
# 6. UTILITIES
# ============================================================

def ensure_output_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


def write_csv(path: str, headers: List[str], rows: List[List[object]]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        f.write(",".join(headers) + "\n")
        for row in rows:
            f.write(",".join(str(v) for v in row) + "\n")


def time_to_within_fraction(series: np.ndarray, times: np.ndarray, frac: float = 0.01) -> float:
    final = series[-1]
    denom = max(abs(final), 1e-12)
    idx = np.where(np.abs(series - final) / denom <= frac)[0]
    return float(times[idx[0]]) if len(idx) else float(times[-1])


# ============================================================
# 7. MAIN ANALYSIS
# ============================================================

def main() -> None:
    ensure_output_dir(P.output_dir)

    l_fine = np.linspace(P.l0, P.lm, 2000)

    print("=" * 68)
    print("NUMERICAL SIMULATION FOR THE SIZE-STRUCTURED FISHERY MODEL")
    print("=" * 68)
    print(f"Using l_m = {P.lm:.1f} cm and L_inf = {P.L_inf:.1f} cm, so g(E,l) > 0 on [l0,lm].")
    print(f"Computed dt = {DT:.6f} yr from CFL condition with safety factor {P.cfl_safety:.2f}.")

    # ------------------------------------------------------
    # No-harvest steady state
    # ------------------------------------------------------
    E_star = find_Estar(l_fine, None, P)
    x_star = stationary_profile(E_star, l_fine, None, P)
    N_star = float(simpson(x_star, x=l_fine))
    R_star = replacement_index(E_star, l_fine, P)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(l_fine, x_star)
    ax.set_xlabel("Body length $l$ (cm)")
    ax.set_ylabel("Stationary density $x^*(l)$")
    ax.set_title("No-Harvest Stationary Size Profile")
    ax.set_xlim(P.l0, P.lm)
    ax.set_ylim(bottom=0)
    ax.annotate(f"$E^*={E_star:.2f}$\n$N^*={N_star:.0f}$\n$R(E^*)={R_star:.3f}$",
                xy=(0.67, 0.78), xycoords="axes fraction",
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="#DDDDDD", alpha=0.85))
    fig.savefig(os.path.join(P.output_dir, "fig_steady_state_profile.pdf"))
    plt.close(fig)

    # ------------------------------------------------------
    # Replacement index curve
    # ------------------------------------------------------
    E_grid = np.linspace(max(1e-3, 0.05 * E_star), 5.0 * E_star, 300)
    R_grid = np.array([replacement_index(E, l_fine, P) for E in E_grid])

    E_crit = np.nan
    if np.any(R_grid > 1.0) and np.any(R_grid < 1.0):
        f = lambda E: replacement_index(E, l_fine, P) - 1.0
        try:
            E_crit = brentq(f, E_grid[0], E_grid[-1])
        except ValueError:
            E_crit = np.nan

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(E_grid, R_grid, label="$R(E)$")
    ax.axhline(1.0, color="#555555", linestyle="--", linewidth=1.5, label="$R=1$")
    ax.axvline(E_star, color="#555555", linestyle=":", linewidth=1.5, label="$E^*$")
    ax.plot([E_star], [R_star], color="#D55E00", marker="o", markersize=7, zorder=5) # Highlight point
    ax.set_xlabel("Crowding level $E$")
    ax.set_ylabel("Intrinsic replacement index $R(E)$")
    ax.set_title("Intrinsic Replacement Index vs. Crowding")
    ax.legend()
    ax.set_ylim(bottom=0)
    fig.savefig(os.path.join(P.output_dir, "fig_replacement_index.pdf"))
    plt.close(fig)

    # ------------------------------------------------------
    # Time dynamics for representative thresholds
    # ------------------------------------------------------
    demo_thresholds = [40.0, 60.0, 80.0]
    demo_results: Dict[float, Dict[str, object]] = {}

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    for l_thr in demo_thresholds:
        res = solve_pde(l_thr, P.T_horizon, p=P, record=True, record_interval=0.25)
        demo_results[l_thr] = res
        ax1.plot(res["times"], res["E_series"], label=fr"$l^*={l_thr:.0f}$ cm")
        ax2.plot(res["times"], res["N_series"], label=fr"$l^*={l_thr:.0f}$ cm")

    no_harvest_dyn = solve_pde(P.lm + 1.0, P.T_horizon, p=P, record=True, record_interval=0.25)
    ax1.plot(no_harvest_dyn["times"], no_harvest_dyn["E_series"], color="#555555", linestyle="--", label="No harvest")
    ax2.plot(no_harvest_dyn["times"], no_harvest_dyn["N_series"], color="#555555", linestyle="--", label="No harvest")

    ax1.set_title("Crowding Index Dynamics")
    ax1.set_xlabel("Time (yr)")
    ax1.set_ylabel("$E(t)$")
    ax1.legend()

    ax2.set_title("Total Population Dynamics")
    ax2.set_xlabel("Time (yr)")
    ax2.set_ylabel("$N(t)$")
    ax2.legend()

    fig.savefig(os.path.join(P.output_dir, "fig_time_dynamics.pdf"))
    plt.close(fig)

    # ------------------------------------------------------
    # Threshold sweep using truncated J_T
    # ------------------------------------------------------
    sweep_thresholds = np.linspace(P.l0 + 2.0, P.lm - 2.0, P.threshold_count)
    Jt_values = []
    R_values = []
    E_ss_values = []
    N_ss_values = []

    x_init_cells = stationary_profile(find_Estar(L_CELLS, None, P), L_CELLS, None, P)

    for idx, l_thr in enumerate(sweep_thresholds, start=1):
        sim = solve_pde(l_thr, P.T_horizon, x_init=x_init_cells, p=P, record=False)
        Jt_values.append(sim["J_T"])

        u_arr_fine = threshold_control(l_thr, l_fine, P)
        E_ss = find_Estar(l_fine, u_arr_fine, P)
        x_ss = stationary_profile(E_ss, l_fine, u_arr_fine, P)
        E_ss_values.append(E_ss)
        N_ss_values.append(float(simpson(x_ss, x=l_fine)))
        R_values.append(replacement_index(E_ss, l_fine, P))

        if idx % 8 == 0 or idx == len(sweep_thresholds):
            print(f"  threshold sweep: {idx}/{len(sweep_thresholds)} complete")

    Jt_values = np.asarray(Jt_values)
    R_values = np.asarray(R_values)
    E_ss_values = np.asarray(E_ss_values)
    N_ss_values = np.asarray(N_ss_values)

    idx_opt = int(np.argmax(Jt_values))
    l_opt = float(sweep_thresholds[idx_opt])
    Jt_opt = float(Jt_values[idx_opt])
    R_opt = float(R_values[idx_opt])
    E_opt_ss = float(E_ss_values[idx_opt])
    N_opt_ss = float(N_ss_values[idx_opt])

    fig, ax = plt.subplots(figsize=(8, 5))
    viable = R_values >= 1.0
    if np.any(~viable):
        ax.fill_between(sweep_thresholds, 0.0, Jt_values.max() * 1.05,
                        where=(~viable), color='#CCCCCC', alpha=0.3, label="$R(E)<1$")
    ax.plot(sweep_thresholds, Jt_values, label=r"$J_T(l^*)$")
    ax.axvline(l_opt, color="#D55E00", linestyle="--", linewidth=1.5, label=fr"$l^*_{{opt}}={l_opt:.1f}$")
    ax.plot([l_opt], [Jt_opt], color="#D55E00", marker="o", markersize=7, zorder=5)
    ax.set_xlabel("Threshold $l^*$ (cm)")
    ax.set_ylabel(r"Truncated discounted revenue $J_T$")
    ax.set_title("Revenue-Threshold Trade-off")
    ax.legend()
    fig.savefig(os.path.join(P.output_dir, "fig_revenue_threshold.pdf"))
    plt.close(fig)

    # ------------------------------------------------------
    # Profile comparison at optimal threshold
    # ------------------------------------------------------
    u_opt_fine = threshold_control(l_opt, l_fine, P)
    x_opt_ss = stationary_profile(E_opt_ss, l_fine, u_opt_fine, P)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(l_fine, x_star, label="No harvest")
    ax.plot(l_fine, x_opt_ss, linestyle="--", label=fr"Optimal threshold $l^*={l_opt:.1f}$ cm")
    ax.axvline(l_opt, color="#555555", linestyle=":", linewidth=1.5, label=fr"$l^*_{{opt}}$")
    ax.set_xlabel("Body length $l$ (cm)")
    ax.set_ylabel("Stationary density")
    ax.set_title("Stationary Profiles: No Harvest vs. Optimal Threshold")
    ax.legend()
    ax.set_xlim(P.l0, P.lm)
    ax.set_ylim(bottom=0)
    fig.savefig(os.path.join(P.output_dir, "fig_profile_comparison.pdf"))
    plt.close(fig)

    # ------------------------------------------------------
    # Numerical summaries and CSV outputs
    # ------------------------------------------------------
    convergence_rows = []
    for l_thr, res in demo_results.items():
        t_E_1pct = time_to_within_fraction(res["E_series"], res["times"], frac=0.01)
        t_N_1pct = time_to_within_fraction(res["N_series"], res["times"], frac=0.01)
        convergence_rows.append([
            l_thr,
            float(res["E_final"]),
            float(res["N_final"]),
            float(res["J_T"]),
            t_E_1pct,
            t_N_1pct,
        ])

    summary_rows = [
        ["l0_cm", P.l0],
        ["lm_cm", P.lm],
        ["L_inf_cm", P.L_inf],
        ["dt_yr", DT],
        ["E_star_no_harvest", E_star],
        ["N_star_no_harvest", N_star],
        ["R_at_E_star", R_star],
        ["E_crit_R_equals_1", E_crit],
        ["l_opt_cm", l_opt],
        ["J_T_opt", Jt_opt],
        ["R_at_opt_steady_state", R_opt],
        ["E_opt_steady_state", E_opt_ss],
        ["N_opt_steady_state", N_opt_ss],
    ]

    write_csv(
        os.path.join(P.output_dir, "numerical_summary.csv"),
        ["quantity", "value"],
        summary_rows,
    )
    write_csv(
        os.path.join(P.output_dir, "threshold_sweep.csv"),
        ["threshold_cm", "J_T", "R_at_ss", "E_ss", "N_ss", "viable_R_ge_1"],
        [[sweep_thresholds[i], Jt_values[i], R_values[i], E_ss_values[i], N_ss_values[i], int(R_values[i] >= 1.0)]
         for i in range(len(sweep_thresholds))],
    )
    write_csv(
        os.path.join(P.output_dir, "dynamics_summary.csv"),
        ["threshold_cm", "E_final", "N_final", "J_T", "time_E_within_1pct", "time_N_within_1pct"],
        convergence_rows,
    )

    print("\n" + "=" * 68)
    print("SUMMARY OF NUMERICAL RESULTS")
    print("=" * 68)
    print(f"No-harvest equilibrium crowding:   E* = {E_star:.6f}")
    print(f"No-harvest total population:       N* = {N_star:.6f}")
    print(f"Replacement index at no-harvest:   R(E*)= {R_star:.6f}")
    if np.isfinite(E_crit):
        print(f"Critical crowding where R(E)=1:    Ecrit= {E_crit:.6f}")
    else:
        print("Critical crowding where R(E)=1:    not bracketed on plotted range")
    print(f"Optimal threshold (truncated J_T): l* = {l_opt:.3f} cm")
    print(f"Maximum truncated revenue:         J_T  = {Jt_opt:.6f}")
    print(f"R at optimal steady state:         R    = {R_opt:.6f}")
    print(f"Optimal steady-state crowding:     E    = {E_opt_ss:.6f}")
    print(f"Optimal steady-state population:   N    = {N_opt_ss:.6f}")
    print(f"Outputs written to: {P.output_dir}")
    print("=" * 68)


if __name__ == "__main__":
    main()

NUMERICAL SIMULATION FOR THE SIZE-STRUCTURED FISHERY MODEL
Using l_m = 130.0 cm and L_inf = 135.3 cm, so g(E,l) > 0 on [l0,lm].
Computed dt = 0.011224 yr from CFL condition with safety factor 0.80.
  threshold sweep: 8/32 complete
  threshold sweep: 16/32 complete
  threshold sweep: 24/32 complete
  threshold sweep: 32/32 complete

SUMMARY OF NUMERICAL RESULTS
No-harvest equilibrium crowding:   E* = 103108.166566
No-harvest total population:       N* = 237004.865496
Replacement index at no-harvest:   R(E*)= 1.474679
Critical crowding where R(E)=1:    Ecrit= 179008.209625
Optimal threshold (truncated J_T): l* = 66.452 cm
Maximum truncated revenue:         J_T  = 1969784.797944
R at optimal steady state:         R    = 1.977621
Optimal steady-state crowding:     E    = 45967.962758
Optimal steady-state population:   N    = 163572.214251
Outputs written to: /mnt/data/fishery_outputs


In [2]:
# ====================================================================
# 8. WEAK-COUPLING APPROXIMATION VALIDATION (FOR REVIEWER 1)
# ====================================================================
import dataclasses

def run_adjoint_validation(l_threshold: float, epsilon: float, p_base: Params = P) -> float:
    """
    Solves the forward PDE under a scaled coupling parameter epsilon, 
    then solves both the full non-local adjoint and reduced adjoint systems 
    backward in time to compute the maximum discrepancy ||lambda - lambda_tilde||.
    """
    # 1. 创建 epsilon 缩放后的参数集
    p_eps = dataclasses.replace(p_base, alpha=epsilon * p_base.alpha, mu1=epsilon * p_base.mu1)
    
    # 重新构建网格参数
    _, l_cells, dl = build_grid(p_eps)
    l_edges = np.linspace(p_eps.l0, p_eps.lm, p_eps.Nl + 1)
    
    # 计算当前参数下的时间步长 dt (确保 CFL 条件稳定)
    max_g = p_eps.K_vb * (p_eps.L_inf - p_eps.l0)
    dt = p_eps.cfl_safety * dl / max_g
    Nt = int(np.ceil(p_eps.T_horizon / dt))
    
    # 设定控制策略
    u_profile = np.where(l_cells > l_threshold, p_eps.u_max, 0.0)
    
    # 2. 正向 PDE 求解：记录每一步的完整轨迹以供逆向伴随使用
    E0 = find_Estar(l_cells, None, p_eps)
    x = stationary_profile(E0, l_cells, None, p_eps).copy()
    
    x_history = []
    E_history = []
    
    for n in range(Nt + 1):
        E = float(simpson(p_eps.chi_coeff * (l_cells**2) * x, x=l_cells))
        x_history.append(x.copy())
        E_history.append(E)
        
        if n == Nt:
            break
            
        g_cent = p_eps.K_vb * (p_eps.L_inf - l_cells) / (1.0 + p_eps.alpha * E)
        mu_cent = (p_eps.mu0 + p_eps.mu1 * E) * np.ones_like(l_cells)
        
        flux = np.zeros(p_eps.Nl + 1)
        flux[0] = p_eps.p_flux
        g_edges = p_eps.K_vb * (p_eps.L_inf - l_edges) / (1.0 + p_eps.alpha * E)
        flux[1:] = g_edges[1:] * x
        
        x_new = x - (dt / dl) * (flux[1:] - flux[:-1]) - dt * (mu_cent + u_profile) * x
        np.maximum(x_new, 0.0, out=x_new)
        x = x_new

    # 3. 逆向 PDE 求解：全伴随 vs 简化伴随
    lambda_full = np.zeros((Nt + 1, p_eps.Nl))
    lambda_red = np.zeros((Nt + 1, p_eps.Nl))
    
    c_vals = p_eps.c0 * (l_cells ** 3)
    chi_vals = p_eps.chi_coeff * (l_cells ** 2)
    
    for n in range(Nt - 1, -1, -1):
        t_next = (n + 1) * dt
        E_next = E_history[n+1]
        x_next = x_history[n+1]
        
        g_next = p_eps.K_vb * (p_eps.L_inf - l_cells) / (1.0 + p_eps.alpha * E_next)
        mu_next = (p_eps.mu0 + p_eps.mu1 * E_next) * np.ones_like(l_cells)
        
        # 共享的基础营收源项
        source_shared = np.exp(-p_eps.r_disc * t_next) * c_vals * u_profile
        
        # --- 简化伴随步 (Reduced Adjoint Step) ---
        lam_red_next = lambda_red[n+1, :]
        lam_red_ext = np.append(lam_red_next[1:], 0.0)  # 边界条件 lambda(lm) = 0
        d_lam_red = (lam_red_ext - lam_red_next) / dl   # 逆向一阶迎风差分
        
        lambda_red[n, :] = lam_red_next + dt * (g_next * d_lam_red - (mu_next + u_profile) * lam_red_next + source_shared)
        
        # --- 全伴随步 (Full Adjoint Step) ---
        lam_full_next = lambda_full[n+1, :]
        lam_full_ext = np.append(lam_full_next[1:], 0.0)
        d_lam_full = (lam_full_ext - lam_full_next) / dl
        
        # 计算偏导数项
        dg_dE = - (p_eps.alpha / (1.0 + p_eps.alpha * E_next)) * g_next
        dmu_dE = p_eps.mu1 * np.ones_like(l_cells)
        
        # 非线性积分反馈项 C(t)
        coupling_integrand = (d_lam_full * dg_dE - lam_full_next * dmu_dE) * x_next
        C_feedback = float(simpson(coupling_integrand, x=l_cells))
        
        source_full = source_shared + chi_vals * C_feedback
        lambda_full[n, :] = lam_full_next + dt * (g_next * d_lam_full - (mu_next + u_profile) * lam_full_next + source_full)
        
    # 返回在整个时空域上的最大绝对误差
    return float(np.max(np.abs(lambda_full - lambda_red)))


def execute_weak_coupling_analysis():
    print("\n" + "=" * 68)
    print("RUNNING WEAK-COUPLING ASYMPTOTIC VALIDATION FOR ADJOINT SYSTEMS")
    print("=" * 68)
    
    epsilons = [0.2, 0.1, 0.05, 0.025, 0.0125]
    errors = []
    
    # 选取基准运行计算出的最优阈值进行检验
    test_threshold = 66.452 
    
    for eps in epsilons:
        err = run_adjoint_validation(test_threshold, eps, P)
        errors.append(err)
        print(f"  Epsilon = {eps:<6} | Max Adjoint Error ||λ - λ_tilde|| = {err:.6e}")
        
    # 计算连续点之间的经验收敛阶 (Error ~ C * eps^q -> q = log(err1/err2)/log(eps1/eps2))
    print("-" * 68)
    print("CONVERGENCE ANALYSIS (Error / Epsilon Ratio):")
    rows = []
    for i, eps in enumerate(epsilons):
        ratio = errors[i] / eps
        rows.append([eps, errors[i], ratio])
        print(f"  Epsilon = {eps:<6} | Error/Epsilon Ratio = {ratio:.6e}")
        
    # 将验证数据导出为 CSV
    ensure_output_dir(P.output_dir)
    write_csv(
        os.path.join(P.output_dir, "weak_coupling_validation.csv"),
        ["epsilon", "max_absolute_error", "error_to_epsilon_ratio"],
        rows
    )
    
    # 绘制双对数误差收敛图验证 O(epsilon)
    fig, ax = plt.subplots(figsize=(7, 5))
    
    # 使用与前面相统一的配色和标记样式 (学术蓝，带圆点标记)
    ax.loglog(epsilons, errors, color="#0072B2", marker='o', markersize=7, linewidth=2, label="Computed Error $\|\lambda - \\tilde{\lambda}\|_{\infty}$")
    
    # 绘制一条标准的 O(epsilon) 渐近参考线 (统一使用深灰色虚线)
    ref_line = [errors[-1] * (e / epsilons[-1]) for e in epsilons]
    ax.loglog(epsilons, ref_line, color="#555555", linestyle="--", linewidth=1.5, label="Theoretical $\\mathcal{O}(\\varepsilon)$ Reference")
    
    ax.set_xlabel("Coupling Parameter $\\varepsilon$")
    ax.set_ylabel("Maximum Adjoint Error")
    ax.set_title("Validation of the Weak-Coupling Approximation")
    
    # 针对 loglog 坐标轴开启 minor 网格辅助线，保持整体风格
    ax.grid(True, which="major", alpha=0.3, linestyle="--")
    ax.grid(True, which="minor", alpha=0.1, linestyle=":")
    
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(P.output_dir, "fig_weak_coupling_error.pdf"))
    plt.close(fig)
    print(f"Weak-coupling plots and data successfully written to {P.output_dir}")
    print("=" * 68)

# 触发该验证函数
execute_weak_coupling_analysis()

<>:136: SyntaxWarning: invalid escape sequence '\|'
<>:136: SyntaxWarning: invalid escape sequence '\|'
C:\Users\lenovo\AppData\Local\Temp\ipykernel_33808\614475849.py:136: SyntaxWarning: invalid escape sequence '\|'
  ax.loglog(epsilons, errors, color="#0072B2", marker='o', markersize=7, linewidth=2, label="Computed Error $\|\lambda - \\tilde{\lambda}\|_{\infty}$")



RUNNING WEAK-COUPLING ASYMPTOTIC VALIDATION FOR ADJOINT SYSTEMS
  Epsilon = 0.2    | Max Adjoint Error ||λ - λ_tilde|| = 2.745849e-01
  Epsilon = 0.1    | Max Adjoint Error ||λ - λ_tilde|| = 1.475677e-01
  Epsilon = 0.05   | Max Adjoint Error ||λ - λ_tilde|| = 7.661884e-02
  Epsilon = 0.025  | Max Adjoint Error ||λ - λ_tilde|| = 3.905397e-02
  Epsilon = 0.0125 | Max Adjoint Error ||λ - λ_tilde|| = 1.971773e-02
--------------------------------------------------------------------
CONVERGENCE ANALYSIS (Error / Epsilon Ratio):
  Epsilon = 0.2    | Error/Epsilon Ratio = 1.372924e+00
  Epsilon = 0.1    | Error/Epsilon Ratio = 1.475677e+00
  Epsilon = 0.05   | Error/Epsilon Ratio = 1.532377e+00
  Epsilon = 0.025  | Error/Epsilon Ratio = 1.562159e+00
  Epsilon = 0.0125 | Error/Epsilon Ratio = 1.577418e+00
Weak-coupling plots and data successfully written to /mnt/data/fishery_outputs


In [3]:
# ====================================================================
# 9. ONE-AT-A-TIME SENSITIVITY ANALYSIS ROUTINE (FOR REVIEWER 2)
# ====================================================================
import dataclasses

def evaluate_parameters(p_sens: Params) -> Tuple[float, float, float, float]:
    """
    Sweeps the threshold values for a given parameter configuration, returning 
    the revenue-maximizing threshold, maximum revenue, steady crowding level, 
    and the steady-state biological replacement index.
    """
    l_fine = np.linspace(p_sens.l0, p_sens.lm, 2000)
    l_cells = L_CELLS  # Keeps consistency with the global spatial grid size (Nl=400)
    
    # 1. Re-evaluate specific initial condition (no-harvest state) for this parameter set
    E0_cells = find_Estar(l_cells, None, p_sens)
    x_init_cells = stationary_profile(E0_cells, l_cells, None, p_sens)
    
    # 2. Re-run threshold sweep over the defined action range
    sweep_thresholds = np.linspace(p_sens.l0 + 2.0, p_sens.lm - 2.0, p_sens.threshold_count)
    Jt_values = []
    
    for l_thr in sweep_thresholds:
        sim = solve_pde(l_thr, p_sens.T_horizon, x_init=x_init_cells, p=p_sens, record=False)
        Jt_values.append(sim["J_T"])
        
    idx_opt = int(np.argmax(Jt_values))
    l_opt = float(sweep_thresholds[idx_opt])
    Jt_opt = float(Jt_values[idx_opt])
    
    # 3. Extract steady-state metrics on the fine grid under the optimal harvesting policy
    u_arr_fine = threshold_control(l_opt, l_fine, p_sens)
    E_ss = find_Estar(l_fine, u_arr_fine, p_sens)
    R_opt = replacement_index(E_ss, l_fine, p_sens)
    
    return l_opt, Jt_opt, E_ss, R_opt


def execute_sensitivity_analysis():
    print("\n" + "=" * 75)
    print("RUNNING ONE-AT-A-TIME SENSITIVITY ANALYSIS FOR REVIEWER 2")
    print("=" * 75)
    
    alpha_vals = [0.0, 2.5e-6, 5.0e-6, 7.5e-6, 1.0e-5]
    r_vals = [0.02, 0.05, 0.08, 0.10]
    l_mat_vals = [40.0, 50.0, 60.0, 70.0]
    
    results_alpha = []
    results_r = []
    results_l_mat = []
    
    print("\n[1/3] Sweeping crowding sensitivity (alpha)...")
    for val in alpha_vals:
        p_sens = dataclasses.replace(P, alpha=val)
        res = evaluate_parameters(p_sens)
        results_alpha.append((val, *res))
        
    print("[2/3] Sweeping economic discount rate (r)...")
    for val in r_vals:
        p_sens = dataclasses.replace(P, r_disc=val)
        res = evaluate_parameters(p_sens)
        results_r.append((val, *res))
        
    print("[3/3] Sweeping maturation length (l_mat)...")
    for val in l_mat_vals:
        p_sens = dataclasses.replace(P, l_mat=val)
        res = evaluate_parameters(p_sens)
        results_l_mat.append((val, *res))
        
    print("\n" + "-" * 75)
    print("LATEX TABLE ROWS FOR DIRECT COPY-PASTE:")
    print("-" * 75)
    
    def fmt_J(j):
        # Format revenue into standard LaTeX scientific notation (\times10^6)
        return f"{j:.4e}".replace("e+06", "\\times10^6")
        
    print("% --- Crowding Sensitivity (alpha) Rows ---")
    for val, l_opt, Jt_opt, E_ss, R_opt in results_alpha:
        if val == 0:
            a_str = "0"
        else:
            if abs(val - 2.5e-6) < 1e-10: a_str = "2.5\\times10^{-6}"
            elif abs(val - 5.0e-6) < 1e-10: a_str = "5.0\\times10^{-6}"
            elif abs(val - 7.5e-6) < 1e-10: a_str = "7.5\\times10^{-6}"
            elif abs(val - 1.0e-5) < 1e-10: a_str = "1.0\\times10^{-5}"
            else: a_str = f"{val:.1e}"
        print(f"\\(\\alpha\\) & \\({a_str}\\) & {l_opt:.2f} & {fmt_J(Jt_opt)} & {E_ss:.2f} & {R_opt:.4f} \\\\")
        
    print("\\hline")
    print("% --- Discount Rate (r) Rows ---")
    for val, l_opt, Jt_opt, E_ss, R_opt in results_r:
        print(f"\\(r\\) & \\({val:.2f}\\) & {l_opt:.2f} & {fmt_J(Jt_opt)} & {E_ss:.2f} & {R_opt:.4f} \\\\")
        
    print("\\hline")
    print("% --- Maturation Length (l_mat) Rows ---")
    for val, l_opt, Jt_opt, E_ss, R_opt in results_l_mat:
        print(f"\\(l_{{\\mathrm{{mat}}}}\\) & \\({int(val)}\\) cm & {l_opt:.2f} & {fmt_J(Jt_opt)} & {E_ss:.2f} & {R_opt:.4f} \\\\")
    print("-" * 75)
    
    # ------------------------------------------------------
    # Plotting the 3-panel Summary Figure
    # ------------------------------------------------------
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4.5))
    
    # Panel (a): alpha vs l_opt
    alphas = [r[0] for r in results_alpha]
    l_opts_a = [r[1] for r in results_alpha]
    # 统一色调：学术蓝，圆点
    ax1.plot(alphas, l_opts_a, color="#0072B2", marker='o', markersize=7, linewidth=2)
    ax1.set_xlabel(r"Crowding Sensitivity $\alpha$")
    ax1.set_ylabel(r"Optimal Threshold $l^*_{\mathrm{opt}}$ (cm)")
    ax1.grid(True, alpha=0.3, linestyle="--")
    ax1.set_title(r"(a) Effect of $\alpha$ on $l^*_{\mathrm{opt}}$")
    
    # Panel (b): r vs l_opt
    rs = [r[0] for r in results_r]
    l_opts_r = [r[1] for r in results_r]
    # 统一色调：学术绿，方块点
    ax2.plot(rs, l_opts_r, color="#009E73", marker='s', markersize=7, linewidth=2)
    ax2.set_xlabel(r"Discount Rate $r$")
    ax2.set_ylabel(r"Optimal Threshold $l^*_{\mathrm{opt}}$ (cm)")
    ax2.grid(True, alpha=0.3, linestyle="--")
    ax2.set_title(r"(b) Effect of $r$ on $l^*_{\mathrm{opt}}$")
    
    # Panel (c): l_mat vs R(E_inf)
    l_mats = [r[0] for r in results_l_mat]
    R_opts = [r[4] for r in results_l_mat]
    # 统一色调：学术橙红，三角点
    ax3.plot(l_mats, R_opts, color="#D55E00", marker='^', markersize=7, linewidth=2)
    # 统一参考线：深灰虚线
    ax3.axhline(1.0, color="#555555", linestyle="--", linewidth=1.5, label=r"$R=1$ Boundary")
    ax3.set_xlabel(r"Maturation Length $l_{\mathrm{mat}}$ (cm)")
    ax3.set_ylabel(r"Replacement Index $R(E_\infty)$")
    ax3.grid(True, alpha=0.3, linestyle="--")
    ax3.legend()
    ax3.set_title(r"(c) Effect of $l_{\mathrm{mat}}$ on $R(E_\infty)$")
    
    fig.tight_layout()
    ensure_output_dir(P.output_dir)
    fig.savefig(os.path.join(P.output_dir, "fig_sensitivity_summary.pdf"))
    plt.close(fig)
    print(f"Successfully generated and saved fig_sensitivity_summary.pdf to {P.output_dir}")
    print("=" * 75)

# Run the sweep
execute_sensitivity_analysis()


RUNNING ONE-AT-A-TIME SENSITIVITY ANALYSIS FOR REVIEWER 2

[1/3] Sweeping crowding sensitivity (alpha)...
[2/3] Sweeping economic discount rate (r)...
[3/3] Sweeping maturation length (l_mat)...

---------------------------------------------------------------------------
LATEX TABLE ROWS FOR DIRECT COPY-PASTE:
---------------------------------------------------------------------------
% --- Crowding Sensitivity (alpha) Rows ---
\(\alpha\) & \(0\) & 76.71 & 2.6802\times10^6 & 59001.41 & 2.3510 \\
\(\alpha\) & \(2.5\times10^{-6}\) & 69.87 & 2.2596\times10^6 & 50255.93 & 2.1435 \\
\(\alpha\) & \(5.0\times10^{-6}\) & 66.45 & 1.9698\times10^6 & 45967.96 & 1.9776 \\
\(\alpha\) & \(7.5\times10^{-6}\) & 59.61 & 1.7602\times10^6 & 38749.29 & 1.8956 \\
\(\alpha\) & \(1.0\times10^{-5}\) & 56.19 & 1.6016\times10^6 & 35258.93 & 1.8090 \\
\hline
% --- Discount Rate (r) Rows ---
\(r\) & \(0.02\) & 66.45 & 3.2967\times10^6 & 45967.96 & 1.9776 \\
\(r\) & \(0.05\) & 66.45 & 1.9698\times10^6 & 45967.96 